In [1]:
from pathlib import Path
import pandas as pd, numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import re
import seaborn as sns

p = Path("2026.02.16_qaoa_out_Novo.xlsx")
df = pd.read_excel(p)
print("Loaded:", p.resolve(), "\nRows, Cols:", df.shape)
print("Has columns:", [c for c in df.columns if c in ["A","D","H","q_Head","q_Center","q_Tail","Orientation in BS"]])

# column setup
orient_col = "Orientation in BS" # pin it explicitly
FILE_STEM = "qaoa_out" # prefix for saved figures

# feature columns like A#, D#, H#
feature_cols = [c for c in df.columns if re.match(r'^[ADH]\d+$', str(c))]
# QAOA columns
qaoa_cols = [c for c in ['QAOA_head','QAOA_center','QAOA_tail'] if c in df.columns]

# summaries
def type_total(prefix):
    cols = [c for c in feature_cols if c.startswith(prefix)]
    return df[cols].sum(axis=1) if cols else pd.Series(0.0, index=df.index)

df['_sum_A_feats'] = type_total('A')
df['_sum_D_feats'] = type_total('D')
df['_sum_H_feats'] = type_total('H')

# long format for feature-level plots
id_col = 'iLigSet' if 'iLigSet' in df.columns else None
long = df.melt(
id_vars=[c for c in [id_col, orient_col] + qaoa_cols if c],
value_vars=feature_cols,
var_name='Feature', value_name='Score'
)
long['Type'] = long['Feature'].str[0]
# treat non-numeric as 0
long['Score'] = pd.to_numeric(long['Score'], errors='coerce').fillna(0.0)
long_nonzero = long[long['Score'] > 0]

# plot helper
sns.set(style="whitegrid", context="talk")
def savefig(name):
    out = f"{FILE_STEM}__{name}.png"
    plt.tight_layout()
    plt.savefig(out, dpi=200, bbox_inches="tight")
    print("saved:", out)
    plt.close()

# 1) Orientation distribution
if orient_col in df.columns:
    plt.figure(figsize=(6,4))
    sns.countplot(x=orient_col, data=df, 
                  order=sorted(df[orient_col].dropna().unique()),
                  color='#002060')  # your custom color here
    plt.title("Orientation counts")
    plt.xlabel("Orientation")
    plt.ylabel("n ligands")
    savefig("orientation_counts")

    
# 2) Feature-level enrichment by Orientation (only if we have nonzeros)
if orient_col in df.columns and not long_nonzero.empty:
    piv = long_nonzero.pivot_table(index='Feature', columns=orient_col,
    values='Score', aggfunc='mean').fillna(0)
    piv['abs_diff'] = piv.max(axis=1) - piv.min(axis=1)
    top = piv.sort_values('abs_diff', ascending=False).head(20).drop(columns='abs_diff')

    # Perceptually-uniform sequential cmap
    cmap = sns.color_palette("crest", as_cmap=True).reversed()
    # Light outlier clipping for better contrast
    vmin = np.percentile(top.values, 2)
    vmax = np.percentile(top.values, 98)

    plt.figure(figsize=(10,8))
    sns.heatmap(top, annot=True, fmt=".2f", cmap=cmap, vmin=vmin, vmax=vmax,
    cbar_kws={"label":"Mean score"})
    plt.title("Top features by Orientation mean difference")
    plt.xlabel("Orientation"); plt.ylabel("Feature")
    savefig("feature_enrichment_heatmap")


# 3) QAOA-selected counts vs Orientation
if orient_col in df.columns and qaoa_cols:
    q = df[[orient_col] + qaoa_cols].melt(id_vars=[orient_col], var_name='Region', value_name='Count')
    # ensure numeric
    q['Count'] = pd.to_numeric(q['Count'], errors='coerce').fillna(0)
    plt.figure(figsize=(8,5))
    sns.barplot(data=q, x='Region', y='Count', hue=orient_col, estimator=np.mean, errorbar='se')
    plt.title("QAOA-selected feature counts by Orientation")
    plt.xlabel("Ligand region"); plt.ylabel("Mean selected features")
    savefig("qaoa_vs_orientation")


# 4) stacked A/D/H composition by Orientation
if orient_col in df.columns and feature_cols:
    bin_feats = df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0).clip(0, 1)

    comp = pd.DataFrame({
        'A': bin_feats[[c for c in feature_cols if c.startswith('A')]].sum(axis=1),
        'D': bin_feats[[c for c in feature_cols if c.startswith('D')]].sum(axis=1),
        'H': bin_feats[[c for c in feature_cols if c.startswith('H')]].sum(axis=1),
        orient_col: df[orient_col].values
    })

    comp_mean = comp.groupby(orient_col)[['A', 'D', 'H']].mean()

    # 100% normalization each row (orientation) is summed to 1
    comp_frac = comp_mean.div(comp_mean.sum(axis=1), axis=0).fillna(0)

    order = ['A', 'D', 'H']
    stack_order = order[::-1]

    palette = {
        'A': '#e57373',
        'D': '#2f89c8',
        'H': '#2ca02c'
    }

    ax = comp_frac[stack_order].plot(kind='bar', stacked=True, figsize=(7, 5),
                                     color=[palette[c] for c in stack_order])

    handles, labels = ax.get_legend_handles_labels()
    hmap = {lab: h for lab, h in zip(labels, handles)}
    ax.legend([hmap[c] for c in order], order, title="Feature", loc="best", frameon=True)

    ax.set_title("A/D/H composition by Orientation (100% stacked)")
    ax.set_xlabel("Orientation")
    ax.set_ylabel("Fraction of present features (sum = 1)")
    savefig("stacked_ADH_by_orientation_100pct")


Loaded: C:\Windows\System32\Docking\New IL feature sets\QAOA for CMG ligands\Novobiocin\Final analysis\2026.02.16_qaoa_out_Novo.xlsx 
Rows, Cols: (21, 28)
Has columns: ['A', 'D', 'H', 'q_Head', 'q_Center', 'q_Tail', 'Orientation in BS']
saved: qaoa_out__orientation_counts.png
saved: qaoa_out__feature_enrichment_heatmap.png
saved: qaoa_out__stacked_ADH_by_orientation_100pct.png
